# Bootstrap spectral benchmarks · adjusted survival and supplementary figure
Refit the midpoint-centered smooth transition for spectral gap and normalized Kemeny using notebook 79's saved patient-level bootstrap particles. The continuous benchmarks come from the original adjusted fits. Both Oλ fits come from the **patient-bootstrap** arm of direct notebook 67. You can run the benchmark fits while notebook 67 runs, then execute the comparison and plotting cells when its patient-bootstrap transition finishes.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
OLD = DATA/'spectral_sensitivity/patient_bayesian_bootstrap_survival_adjusted'
INPUT = OLD/'combined_inputs'
EST = INPUT/'patient_spectral_benchmark_estimates.csv'
NPZ = INPUT/'patient_spectral_benchmark_particles.npz'
CLINICAL = DATA/'clinical_outcomes.csv'
CONT = OLD/'continuous'
SENS = DATA/'bayesian_Olambda_sensitivity_survival/direct_reciprocal_adjusted_midpoint_v1'
OUTPUT_DIR = DATA/'bootstrap_spectral_survival_comparison_direct_midpoint_v1'
FEATURES = ['spectral_gap','normalized_kemeny_constant']
ADJUST = {'Treatment':'axicel','Disease':'lymphoma'}
transition = 'smooth_transition_Q10_Q90_midpoint'
def read_json(path):
    return json.loads(path.read_text())

In [ ]:
REVISION = '43d2dc51d423a0e3ceacdc2564787a73b31d1fe9'
repo = Path(tempfile.mkdtemp(prefix='direct-survival-')) / 'repo'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'direct-olambda',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--quiet', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymc==5.28.5',
                'arviz>=0.22,<1', 'h5netcdf>=1.5'], check=True)
print('Survival model revision:', REVISION)

In [ ]:
# Confirm that the saved particles reproduce the benchmark central estimates.
keys = ['Dataset','sample_id']
est = pd.read_csv(EST).set_index(keys,verify_integrity=True)
with np.load(NPZ,allow_pickle=False) as archive:
    names = archive['feature_names'].astype(str).tolist()
    ids = pd.MultiIndex.from_arrays([archive['datasets'].astype(str),
        archive['patient_ids'].astype(str)],names=keys)
    values = archive['values'][:,:,[names.index(f) for f in FEATURES]]
    weights = archive['particle_weights']
assert ids.is_unique and ids.sort_values().equals(est.index.sort_values())
assert values.shape==(len(ids),25,2) and np.isfinite(values).all()
np.testing.assert_allclose(weights,np.full(values.shape[:2],1/25),rtol=1e-6)
np.testing.assert_allclose(est.loc[ids,FEATURES],values.mean(axis=1,dtype=float),rtol=1e-5)
old_manifest = read_json(CONT/'individual_run_manifest.json')
assert old_manifest['adjustments']==ADJUST
continuous = pd.read_csv(CONT/'individual_results.csv').set_index('feature',verify_integrity=True).loc[FEATURES].reset_index()
def cohort(folder):
    fields = ['event_observed','t_start','t_end',*ADJUST]
    return pd.read_csv(folder/'analysis_cohort.csv',usecols=keys+fields).set_index(keys,verify_integrity=True).sort_index()
baseline = cohort(CONT)
assert len(baseline)==163 and baseline.index.get_level_values('Dataset').nunique()==8
outcomes = pd.read_csv(CLINICAL)
if 'event_observed' not in outcomes:
    outcomes = outcomes.rename(columns={'event':'event_observed'})
clinical_rows = outcomes.set_index(keys,verify_integrity=True).loc[baseline.index,baseline.columns]
pd.testing.assert_frame_equal(baseline,clinical_rows,check_dtype=False)
print('Benchmark patients:',len(baseline),'| particles per patient:',values.shape[1])
display(continuous[['feature','HR_median','HR_q025','HR_q975']])

In [ ]:
args = ['--midpoint-centered','--crossover-quantile-low','0.10','--crossover-quantile-high','0.90',
        '--transition-scale-sd','0.10','--minimum-cells','1000','--draws','2000','--tune','2000',
        '--chains','4','--cores','4','--target-accept','0.99','--random-state','0']
for col,reference in ADJUST.items():
    args.extend(['--adjustment',f'{col}={reference}'])
for feature in FEATURES:
    args.extend(['--feature',feature])
config = dict(revision=REVISION,arguments=args,
              input_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [EST,NPZ,CLINICAL]})
tag = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:12]
FIT = OUTPUT_DIR/'benchmark_transition'/tag
FIT.mkdir(parents=True,exist_ok=True)
(FIT/'notebook_config.json').write_text(json.dumps(config,indent=2))
command = [sys.executable,'-u','-m','survival.fit_tcat_survival_regimes',EST,NPZ,CLINICAL,FIT,*args]
print('Fitting spectral gap and normalized Kemeny. Completed feature fits resume from',FIT,flush=True)
with subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
    for line in proc.stdout:
        print(line,end='',flush=True)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode,command)
if not (FIT/'regime_results.csv').is_file():
    raise RuntimeError(f'No completed benchmarks. Check {FIT}/regime_errors.csv')
regimes = pd.read_csv(FIT/'regime_results.csv')
if regimes.feature.duplicated().any() or set(regimes.feature)!=set(FEATURES):
    raise ValueError(f'Incomplete benchmark transitions. Check {FIT}/regime_errors.csv and rerun this cell.')
assert regimes.hazard_activation_offset.eq(.5).all()
pd.testing.assert_frame_equal(baseline,cohort(FIT),check_dtype=False)
display(regimes[['feature','HR_median','HR_q025','HR_q975','crossover_median','divergences','max_rhat']])

## Assemble the six estimates
Continuous HRs are per global SD of the predictor. Transition HRs compare the high-value plateau with the low-value plateau for each statistic. The original orientation of spectral gap and normalized Kemeny is retained.

In [ ]:
module_index = pd.read_csv(SENS/'sensitivity_survival_model_comparison.csv')
patient = module_index.loc[module_index.experiment.eq('patient-bootstrap')].set_index('fit',verify_integrity=True)
feature_order = ['O_lambda_bootstrap',*FEATURES]
model_order = ['Continuous','Smooth transition']
rows = [continuous.assign(model=model_order[0],source=str(CONT/'individual_results.csv')),
        regimes.assign(model=model_order[1],source=str(FIT/'regime_results.csv'))]
for fit,label in zip(['continuous',transition],model_order):
    if fit not in patient.index:
        raise ValueError(f'Waiting for notebook 67 patient-bootstrap {fit}. Rerun this cell after completion.')
    root = SENS/'patient_bayesian_bootstrap'/fit/Path(patient.loc[fit,'output_dir']).name
    m = read_json(root/'run_manifest.json')
    assert m['adjustments']==ADJUST and m['n_particles']==25
    if fit==transition:
        assert m['hazard_activation_offset']==.5
    pd.testing.assert_frame_equal(baseline,cohort(root),check_dtype=False)
    source = root/'model_result.csv'
    result = pd.read_csv(source)
    assert len(result)==1
    rows.append(result.assign(feature='O_lambda_bootstrap',model=label,source=str(source)))
comparison = pd.concat(rows,ignore_index=True)
expected = pd.MultiIndex.from_product([feature_order,model_order],names=['feature','model'])
comparison = comparison.set_index(['feature','model'],verify_integrity=True).loc[expected].reset_index()
hr = comparison[['HR_q025','HR_median','HR_q975']].to_numpy(float)
assert np.isfinite(hr).all() and (hr>0).all() and (np.diff(hr,axis=1)>=0).all()
assert comparison.patients.eq(len(baseline)).all() and comparison.events.eq(baseline.event_observed.sum()).all()
comparison.to_csv(OUTPUT_DIR/'bootstrap_spectral_survival_comparison.csv',index=False)
display(comparison[['feature','model','HR_median','HR_q025','HR_q975','P_all_datasets_same_direction',
                    'divergences','max_rhat','min_ess_bulk']])

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'pdf.fonttype':42})

feature_labels = {
    'O_lambda_bootstrap': r'$\mathcal{O}_{\lambda}$ bootstrap',
    'spectral_gap': 'Spectral gap bootstrap',
    'normalized_kemeny_constant': 'Normalized Kemeny bootstrap',
}
positions = np.arange(len(feature_order), dtype=float)
offsets = {'Continuous': -0.13, 'Smooth transition': 0.13}
colors = {'Continuous': '#3B6EA8', 'Smooth transition': '#B84A4A'}
markers = {'Continuous': 'o', 'Smooth transition': 's'}

fig, axis = plt.subplots(figsize=(7.6, 3.8))
for model in model_order:
    subset = (
        comparison.loc[comparison.model.eq(model)]
        .set_index('feature').loc[feature_order]
    )
    y = positions + offsets[model]
    median = subset.HR_median.to_numpy(float)
    lower = subset.HR_q025.to_numpy(float)
    upper = subset.HR_q975.to_numpy(float)
    axis.errorbar(
        median, y,
        xerr=np.vstack([median - lower, upper - median]),
        fmt=markers[model], markersize=6.2,
        color=colors[model], ecolor=colors[model],
        elinewidth=1.35, capsize=3.0, capthick=1.1,
        label=model, zorder=3,
    )

axis.axvline(1, color='#555555', linestyle='--', linewidth=1.0, zorder=1)
axis.set_xscale('log', base=2)

# Preserve the reciprocal visual spacing of the log2 axis while showing ordinary
# hazard-ratio labels rather than 2^-1, 2^0, and 2^1 notation.
xmin = min(0.4, float(comparison.HR_q025.min()) / 1.10)
xmax = max(4.5, float(comparison.HR_q975.max()) * 1.10)
ticks = 2.0 ** np.arange(np.ceil(np.log2(xmin)), np.floor(np.log2(xmax))+1)
axis.set_xticks(ticks, [f'{value:g}' for value in ticks])
axis.xaxis.set_minor_formatter(NullFormatter())
axis.set_xlim(xmin,xmax)

axis.set_yticks(positions, [feature_labels[feature] for feature in feature_order])
axis.invert_yaxis()
axis.set_xlabel('Hazard ratio')
axis.grid(axis='x', which='major', color='#E5E5E5', linewidth=0.7)
axis.set_axisbelow(True)
axis.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0))
for spine in ['top', 'right', 'left']:
    axis.spines[spine].set_visible(False)
axis.tick_params(axis='y', length=0, pad=9)

fig.subplots_adjust(left=0.30, right=0.79, top=0.96, bottom=0.19)

jpg_1200 = OUTPUT_DIR / 'bootstrap_spectral_survival_comparison_1200dpi.jpg'
jpg_600 = OUTPUT_DIR / 'bootstrap_spectral_survival_comparison_600dpi.jpg'
pdf_path = OUTPUT_DIR / 'bootstrap_spectral_survival_comparison.pdf'
fig.savefig(jpg_1200, dpi=1200, bbox_inches='tight', facecolor='white',
            pil_kwargs={'quality': 95})
fig.savefig(jpg_600, dpi=600, bbox_inches='tight', facecolor='white',
            pil_kwargs={'quality': 95})
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)

print('Saved:', jpg_1200)
print('Saved:', jpg_600)
print('Saved:', pdf_path)